# DAO Treasury: computational-science simulation

COMSCI/ECON 206 PS2 · Zhou Zimu (FP11). This notebook executes the same standard-library simulation as `python -m src.simulation` and saves the three committed outputs.

## Model assumptions

One whale (60% stake) and seven ordinary holders (40% total) choose one of three mutually exclusive proposals. Every actor sincerely votes for the proposal with their largest valuation. Valuations are independently uniform on [0, 1), drawn once with seed 206 and reused across mechanisms. Weighted plurality selects the largest score, breaking exact ties A then B then C. SRV transforms each **wallet** stake with its square root; eight Sybil wallets split the whale stake equally and coordinate on the whale’s favorite. Eight economic actors remain eight actors. The browser-based self-play interface is separate from these random profiles.

In [1]:
from pathlib import Path
import csv
import json
from src.simulation import CONDITIONS, METRICS, load_parameters, run, save

root = Path.cwd()
assert (root / 'src' / 'simulation.py').exists(), 'Run from the complete repository root with src/ and inputs/ available.'
p = load_parameters()
print(json.dumps(p, indent=2))

{
  "seed": 206,
  "profiles": 1000,
  "total_actors": 8,
  "ordinary_holders": 7,
  "whale_holders": 1,
  "whale_token_share": 0.6,
  "ordinary_collective_share": 0.4,
  "proposals": [
    "A",
    "B",
    "C"
  ],
  "sybil_wallets": 8,
  "valuation_distribution": "independent uniform [0, 1) per actor and proposal",
  "tie_break": "proposal order A, B, C"
}


## Simulation procedure

For each of 1,000 paired profiles, draw eight-by-three valuations, compute each actor’s top proposal, then score TWV, one-wallet SRV, and eight-wallet SRV. The whale’s Sybil identities retain the same aggregate stake and preference. The comparison averages each per-profile metric.

In [2]:
rows, comparison = run(p)
assert len(rows) == 3 * p['profiles']
assert [c['condition'] for c in comparison] == [c[0] for c in CONDITIONS]
assert all(rows[i]['profile_id'] == rows[i+1]['profile_id'] == rows[i+2]['profile_id'] for i in range(0, len(rows), 3))
assert all(rows[i]['winner'] == rows[i+2]['winner'] for i in range(0, len(rows), 3))
print(f"{p['profiles']} paired profiles; {len(rows)} condition-profile rows")
print('TWV and eight-wallet SRV winners agree in all 1,000 profiles')

1000 paired profiles; 3000 condition-profile rows
TWV and eight-wallet SRV winners agree in all 1,000 profiles


## Metric definitions and mechanism comparison

Holder efficiency = winning total actor valuation divided by best total actor valuation, averaged across profiles. Stake efficiency applies actual token shares to actor valuations before taking that ratio. Whale success is the fraction where the whale’s favorite wins. Non-whale representation is the fraction where the winner maximizes the seven ordinary actors’ **combined** valuation; it does not count seven separate wins.

In [3]:
print('condition       holder_eff  stake_eff   whale_success  nonwhale_rep')
for c in comparison:
    print(f"{c['condition']:<15} {c['holder_efficiency']:.6f}     {c['stake_efficiency']:.6f}     {c['whale_success']:.3f}          {c['nonwhale_representation']:.3f}")
print('SRV one wallet minus TWV, non-whale representation:', round(comparison[1]['nonwhale_representation'] - comparison[0]['nonwhale_representation'], 3))

condition       holder_eff  stake_eff   whale_success  nonwhale_rep
TWV_m1          0.899065     0.991097     1.000          0.321
SRV_m1          0.927781     0.977076     0.896          0.417
SRV_m8          0.899065     0.991097     1.000          0.321
SRV one wallet minus TWV, non-whale representation: 0.096


## Save and inspect outputs

The detailed CSV includes each outcome and per-profile metric. Valuations can be regenerated from the committed seed and generator; they are not duplicated in the CSV. The comparison CSV and summary JSON are calculated from those same rows.

In [4]:
summary = save(rows, comparison, p)
with open(root / 'outputs' / 'simulation_results.csv', newline='', encoding='utf-8') as f:
    detailed = list(csv.DictReader(f))
with open(root / 'outputs' / 'mechanism_comparison.csv', newline='', encoding='utf-8') as f:
    stored = list(csv.DictReader(f))
with open(root / 'outputs' / 'summary.json', encoding='utf-8') as f:
    disk_summary = json.load(f)
assert len(detailed) == 3000 and len(stored) == 3
assert disk_summary == summary
for observed in stored:
    for metric in METRICS:
        assert abs(float(observed[metric]) - summary['results'][observed['condition']][metric]) < 1e-12
print('Saved and re-read 3,000 detailed rows, 3 comparison rows, and a matching summary JSON.')
print('First profile winners:', [(r['condition'], r['winner']) for r in detailed[:3]])

Saved and re-read 3,000 detailed rows, 3 comparison rows, and a matching summary JSON.
First profile winners: [('TWV_m1', 'C'), ('SRV_m1', 'C'), ('SRV_m8', 'C')]


## Interpretation

With one wallet, SRV increases non-whale representation from 0.321 to 0.417 and holder efficiency from 0.899065 to 0.927781 in this sample. Stake efficiency declines from 0.991097 to 0.977076. Splitting 0.60 into eight equal SRV wallets gives the whale total weight √(8 × 0.60) ≈ 2.190890, which exceeds all seven ordinary holders’ combined weight ≈ 1.673320. The whale therefore determines all outcomes in that condition, just as in TWV here. This is a paired synthetic comparison, not evidence from participants or a claim of strategic equilibrium.

## Limitations

The result is conditional on one seed and 1,000 profiles, uniform independent valuations, sincere votes, equal ordinary stakes, three alternatives, and zero-cost coordinated Sybil identities. Different distributions, voting behavior, identity checks, abstentions, or treasury outcomes can change the magnitudes. The page’s older numbers lacked source code or a specified stochastic model and are reconciled in `VERIFICATION.md`.